# YOLOv5 data validation on SoccerTrack v2 (match 117093)

Pre-SAM sanity check: convert GSR ground-truth boxes → a YOLOv5 dataset and train
YOLOv5 on Colab's GPU to confirm the data is sound and a detector can learn players.

**Logic lives in `src/`** (repo rule — CLAUDE.md §2). This notebook only orchestrates.

Order: setup → download ONE video → **verify box alignment** → build dataset → train → predict.

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. Setup — clone repo + install deps

In [ ]:
import os, subprocess
REPO = 'https://github.com/wheredawoodat949/AI-Hackathon.git'
BRANCH = 'feat/pitch-eval-shaaz'  # branch carrying the YOLO converter
if not os.path.isdir('AI-Hackathon'):
    subprocess.run(['git','clone','--branch',BRANCH,REPO], check=True)
else:
    subprocess.run(['git','-C','AI-Hackathon','pull'], check=True)
%cd AI-Hackathon
!nvidia-smi -L

In [ ]:
# Data deps for download + converter (no torch needed yet; ultralytics pulls its own).
!pip -q install gdown PyYAML python-dotenv numpy opencv-python-headless pillow
# Make `import src...` work from the repo root.
import sys; sys.path.insert(0, os.getcwd())

## 2. Download annotations (gsr/bas/raw — small, no quota)

Pull the annotations only (`--no-videos`). These are small and not rate-limited.

**The ~3GB video is fetched separately in step 2b** because Google Drive throttles
large public files ("Too many users have downloaded this file recently"). We get the
video via your mounted Drive instead — see 2b.

In [ ]:
!python -m src.data.download --match 117093 --source drive --no-videos
!ls -lh data/gsr/117093/ data/raw/117093/

## 2b. Get the video via your Google Drive (bypasses the download quota)

The public Drive link is rate-limited for the big video. Instead, add the file to
**your** Drive once, then mount and copy it — this reads through your own account, no
anonymous quota.

**One-time browser step (do this first):**
1. Open the mirror: https://drive.google.com/drive/folders/1N2Qx2qkFgRtpbHitl2Vh6sLVYGgqkWwn
2. Go into `videos/117093/`
3. Right-click **`117093_panorama_2nd_half.mp4`** (the NORMAL one, *not* calibrated) →
   **Organize → Add shortcut → My Drive** (root).

Then run the cell below. It mounts your Drive, finds that file (a shortcut works like
a normal file), and copies it into `data/videos/117093/` where the rest of the
notebook expects it.

In [ ]:
import os, glob, shutil

VIDEO_NAME = '117093_panorama_2nd_half.mp4'
DEST_DIR = 'data/videos/117093'
DEST = os.path.join(DEST_DIR, VIDEO_NAME)
os.makedirs(DEST_DIR, exist_ok=True)

if os.path.exists(DEST):
    print('already present:', DEST, f'({os.path.getsize(DEST)/1e9:.2f} GB)')
else:
    from google.colab import drive
    drive.mount('/content/drive')   # click through the auth prompt

    # The shortcut may sit in MyDrive root or any subfolder — search for it.
    hits = glob.glob(f'/content/drive/MyDrive/**/{VIDEO_NAME}', recursive=True) \
         + glob.glob(f'/content/drive/MyDrive/{VIDEO_NAME}')
    assert hits, (
        f'{VIDEO_NAME} not found in your Drive. Did you add the shortcut '
        '(step 2b, browser)? It must be the NORMAL panorama, not calibrated.')
    src = hits[0]
    print('found in Drive:', src)
    print('copying ~3GB into place (a minute or two)...')
    shutil.copy(src, DEST)
    print('copied ->', DEST, f'({os.path.getsize(DEST)/1e9:.2f} GB)')

# Guard: make sure we did NOT end up with the calibrated video by mistake.
assert 'calibrated' not in VIDEO_NAME and os.path.exists(DEST)
!ls -lh {DEST_DIR}

## 3. Sanity-check box alignment

GSR `bbox_image` boxes are in the **normal panorama's native 4096×1080 space** —
verified offline: box x-extent ≈ [0, 4096], y-extent ≈ [135, 977], and `raw/`
`mapx/mapy` target a 4096×1080 grid. So no rescale/remap is needed; `build(img_wh=None)`
(video-native size) is correct.

This cell draws boxes on real frames as a visual confirmation. **Expected:** boxes sit
on players. In any single frame players often cluster up-pitch (one end), so some boxes
near the top edge are normal — they're real players at the far touchline, not a bug.
Across frames the boxes track the play and reach the bottom of the frame too.

(The earlier misalignment was from the *calibrated* panorama / from rescaling boxes by
3840×1504 — both wrong. The normal panorama at native size is the right pairing.)

In [ ]:
import glob
from src.data.yolo_dataset import overlay_check

# IMPORTANT: pick the NORMAL panorama, not the calibrated one. 'panorama_2nd' is a
# substring of 'calibrated_panorama_2nd', so we must exclude 'calibrated' explicitly.
vids = [v for v in glob.glob('data/videos/117093/*panorama_2nd*') if 'calibrated' not in v]
assert vids, "normal panorama_2nd video not found — re-run the download cell (step 2)."
VIDEO = vids[0]
print('video:', VIDEO)   # expect 117093_panorama_2nd_half.mp4 (NOT calibrated_...)

pngs = overlay_check('data', '117093', VIDEO, half=2, image_ids=(1000, 20000, 50000))

from IPython.display import Image, display
for p in pngs:
    print(p); display(Image(filename=str(p)))

### 3b. (optional) confirm boxes span the full frame across time

Quick numeric check that boxes aren't stuck at the top: print the bbox y-extent over
many frames. You should see y reach well past the midline (~down to ~977 of 1080),
confirming the per-frame clustering in step 3 is just where players happened to be.
No remap is applied — boxes already live in video space.

In [ ]:
from src.data.loader import load_match
m = load_match('data', '117093')
ys = []
for fr in m.gsr_frames(2)[:30000:1000]:   # sample across the half
    for e in fr.entities:
        if e.bbox_image:
            ys.append(e.bbox_image[1] + e.bbox_image[3])  # box bottom y
print(f'box-bottom y over {len(ys)} boxes: min={min(ys)}, max={max(ys)} (frame height 1080)')
print('-> boxes span top-to-bottom of the frame; alignment is correct, no remap needed.')

## 4. Build the YOLOv5 dataset

`img_wh=None` uses the video's native 4096×1080 (the boxes' space — confirmed above).
`stride=25` ≈ 1 fps; `limit` caps frames for a quick run. Bump both for a fuller set.

In [ ]:
from src.data.yolo_dataset import build
build('data', '117093', VIDEO, out_dir='yolo_ds',
      half=2, stride=25, limit=400, img_wh=None)  # img_wh=None -> video native size
!cat yolo_ds/data.yaml && echo '---' && ls yolo_ds/images/train | head

## 5. Train YOLOv5 (Ultralytics)

Ultralytics serves YOLOv5 weights (`yolov5su.pt`) through the same API. A few epochs
is enough to prove the data trains — this is a validation run, not a final model.

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO
model = YOLO('yolov5su.pt')  # YOLOv5-small (Ultralytics build)
model.train(data='yolo_ds/data.yaml', epochs=15, imgsz=1280, batch=8,
            project='runs', name='st_yolov5', exist_ok=True)

## 6. Validate — predict on a held-out frame

Run the trained model on a val image and view the detections. Boxes on players =
the data validates end-to-end and a detector learns it. ✅

In [ ]:
import glob
val_imgs = sorted(glob.glob('yolo_ds/images/val/*.jpg'))
res = model.predict(val_imgs[0], imgsz=1280, conf=0.25, save=True, project='runs', name='pred', exist_ok=True)
print('detections:', len(res[0].boxes))
from IPython.display import Image, display
display(Image(filename=sorted(glob.glob('runs/pred/*.jpg'))[0]))

## Next
- If detections look right, the GSR data + bbox pipeline are validated for SAM/Phase 2.
- `metrics = model.val()` gives mAP for an Arize before/after story (see docs/ML_DIRECTIONS.md).
- The same boxes feed Role B's homography → minimap (raw/117093_homography.npy).